In [1]:
%load_ext autoreload
%autoreload 2

# 05 — Search 模块测试

覆盖 `backend/search/` 的全部搜索方法：
1. 基础搜索层：LocalSearch、GlobalSearch
2. 工具封装层：LocalSearchTool、GlobalSearchTool、HybridSearchTool、NaiveSearchTool
3. 高级搜索：DeepResearchTool、DeeperResearchTool
4. 专用工具：ChainOfExplorationTool、HypothesisGeneratorTool、AnswerValidationTool
5. 工具注册表：tool_registry
6. 数据适配：retrieval_adapter
7. 向量工具：VectorUtils

**前置条件**：Neo4j 已启动，知识库中已有数据（Entity、Chunk、Community 节点）。

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from backend.config.neo4jdb import get_db_manager
from backend.models.get_models import get_llm_model, get_embeddings_model
from backend.config.settings import (
    BASE_SEARCH_CONFIG,
    LOCAL_SEARCH_SETTINGS,
    GLOBAL_SEARCH_SETTINGS,
    HYBRID_SEARCH_SETTINGS,
    NAIVE_SEARCH_TOP_K,
)

print("环境检查:")
neo4j_info = get_db_manager()
print(f"  Neo4j: {neo4j_info.neo4j_uri} (connected)")

llm = get_llm_model()
embeddings = get_embeddings_model()
print(f"  LLM: {llm.__class__.__name__}")
print(f"  Embeddings: {embeddings.__class__.__name__}")
print()
print("✓ 初始化完成")

g:\system1_software\anaconda\envs\mpcrl\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


环境检查:
  Neo4j: bolt://localhost:7687 (connected)


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2594.20it/s]


[embedding_worker] Model loaded: shibing624/text2vec-base-chinese
  LLM: ChatOpenAI
  Embeddings: _SentenceTransformerAdapter

✓ 初始化完成


---
## 2. 基础搜索测试

### 2.1 LocalSearch — 本地搜索

In [2]:
from backend.search.local_search import LocalSearch

local = LocalSearch(llm, embeddings)

# 测试 as_retriever
retriever = local.as_retriever()
print(f"检索器类型: {type(retriever).__name__}")
print(f"top_entities={local.top_entities}, top_chunks={local.top_chunks}")

# 测试 search
query = "设备故障原因分析"
print(f"\n查询: {query}")
result = local.search(query)
print(f"\n回答: {result}..." if len(result) > 300 else f"\n回答: {result}")

检索器类型: VectorStoreRetriever
top_entities=10, top_chunks=3

查询: 设备故障原因分析

=== 向量检索耗时: 0.247s ===

--- doc[0] ---
Entities:
- {'id': '数控系统冷却单元故障', 'description': '故障代码E-0123，导致主轴驱动电机过热报警。', 'type': '故障'}
- {'id': '水垢沉积', 'description': '主轴冷却循环管路长期未清洗导致内部水垢沉积。', 'type': '故障原因'}
- {'id': '主轴温度异常升高至95℃', 'description': '主轴连续运行4小时后温度异常升高至95℃，触发过热保护停机。', 'type': '故障现象'}
- {'id': '主轴驱动电机', 'description': '驱动主轴运转的电机，是数控加工中心核心驱动部件。', 'type': '部件'}
- {'id': '拆解主轴冷却系统并循环清洗', 'description': '拆解主轴冷却系统，使用专业除垢剂对冷却管路进行循环清洗。', 'type': '维修措施'}
- {'id': '数控系统冷却单元', 'description': '数控加工中心数控系统的冷却单元，负责冷却系统散热。', 'type': '部件'}
- {'id': '数控加工中心', 'description': '型号VMC-850，用于精密零件加工的高精度数控设备。', 'type': '设备'}
- {'id': '冷却液添加剂浓度不足', 'description': '冷却液添加剂浓度不足，导致管路腐蚀加剧。', 'type': '故障原因'}
- {'id': '重新调配冷却液至标准浓度并调试', 'description': '重新调配冷却液至标准浓度，并进行调试和核实。', 'type': '维修措施'}
- {'id': '主轴冷却循环管路', 'description': '主轴冷却系统的重要组成部分，用于循环冷却液。', 'type': '部件'}
Reports:
Chunks:
- {'id': '89a1e97ba75e4ad7b8ba628dcf1749dd8be83481', 't

### 2.2 GlobalSearch — 全局搜索

In [ ]:
from backend.search.global_search import GlobalSearch

global_s = GlobalSearch(llm)

query = "设备故障原因分析"
print(f"查询: {query}")
print("正在执行 Map-Reduce 全局搜索...")
result = global_s.search(query, level=0)
print(f"\n回答: {result[:300]}..." if len(result) > 300 else f"\n回答: {result}")

查询: 有哪些设备故障类型
正在执行 Map-Reduce 全局搜索...


正在处理社区数据: 0it [00:00, ?it/s]


回答: 不知道。


---
## 3. 工具封装层测试

### 3.1 LocalSearchTool

In [ ]:
from backend.search.tool.local_search_tool import LocalSearchTool

ls_tool = LocalSearchTool()

# 测试关键词提取
keywords = ls_tool.extract_keywords("电机过热故障原因")
print(f"关键词提取: {keywords}")

# 测试基础搜索
result = ls_tool.search("轴承磨损检测方法有哪些？")
print(f"\n搜索回答: {result[:300]}..." if len(result) > 300 else f"\n搜索回答: {result}")

# 测试结构化搜索
structured = ls_tool.structured_search("轴承磨损检测方法")
print(f"\n结构化结果字段: {list(structured.keys())}")
print(f"检索结果数量: {len(structured.get('retrieval_results', []))}")
print(f"raw_context 数量: {len(structured.get('raw_context', []))}")

# 测试 Tool 封装
tool = ls_tool.get_tool()
print(f"\nTool name: {tool.name}")
print(f"Tool description: {tool.description[:50]}...")

structured_tool = ls_tool.get_structured_tool()
print(f"StructuredTool name: {structured_tool.name}")

print("\n✓ LocalSearchTool 测试完成")

### 3.2 GlobalSearchTool

In [ ]:
from backend.search.tool.global_search_tool import GlobalSearchTool

gs_tool = GlobalSearchTool(level=0)

# 测试关键词提取
keywords = gs_tool.extract_keywords("常见的设备故障有哪些")
print(f"关键词: {keywords}")

# 测试搜索
result = gs_tool.search("常见的设备故障有哪些")
print(f"\n中间结果数: {len(result) if isinstance(result, list) else 'N/A'}")

# 测试结构化搜索
structured = gs_tool.structured_search("常见的设备故障有哪些")
print(f"\n结构化结果字段: {list(structured.keys())}")
print(f"final_answer: {structured.get('final_answer', '')[:200]}...")
print(f"检索结果数量: {len(structured.get('retrieval_results', []))}")

print("\n✓ GlobalSearchTool 测试完成")

### 3.3 HybridSearchTool

In [ ]:
from backend.search.tool.hybrid_tool import HybridSearchTool

hybrid = HybridSearchTool()

# 测试关键词提取
keywords = hybrid.extract_keywords("电机过热故障原因及解决方案")
print(f"关键词:\n  low_level: {keywords.get('low_level', [])}\n  high_level: {keywords.get('high_level', [])}")

# 测试混合搜索
result = hybrid.search("电机过热故障原因及解决方案")
print(f"\n搜索回答: {result[:300]}..." if len(result) > 300 else f"\n搜索回答: {result}")

# 测试结构化搜索
structured = hybrid.structured_search("电机过热故障原因及解决方案")
print(f"\n结构化结果字段: {list(structured.keys())}")
print(f"final_answer: {structured.get('final_answer', '')[:200]}...")
print(f"low_level_content 长度: {len(structured.get('low_level_content', ''))}")
print(f"high_level_content 长度: {len(structured.get('high_level_content', ''))}")
print(f"检索结果数量: {len(structured.get('retrieval_results', []))}")

# 测试性能指标
print(f"\n性能指标: {hybrid.performance_metrics}")

print("\n✓ HybridSearchTool 测试完成")

### 3.4 NaiveSearchTool

In [ ]:
from backend.search.tool.naive_search_tool import NaiveSearchTool

naive = NaiveSearchTool()

print(f"NAIVE_SEARCH_TOP_K = {NAIVE_SEARCH_TOP_K}")

result = naive.search("设备停机故障")
print(f"\n搜索回答: {result[:300]}..." if len(result) > 300 else f"\n搜索回答: {result}")

print(f"\n性能指标: {naive.performance_metrics}")

print("\n✓ NaiveSearchTool 测试完成")

---
## 4. 高级搜索测试

### 4.1 DeepResearchTool（深度研究）

> 注意：DeepResearch 会执行多轮 LLM 调用，耗时较长。可调整 `MAX_SEARCH_LIMIT` 控制迭代轮数。

In [ ]:
import time
from backend.search.tool.deep_research_tool import DeepResearchTool

dr_tool = DeepResearchTool()

# 可选：限制迭代轮数以加速测试
dr_tool.max_iterations = 2

query = "设备故障的主要原因有哪些？"
print(f"查询: {query}")
print(f"最大迭代: {dr_tool.max_iterations}")

start = time.time()
result = dr_tool.search(query)
elapsed = time.time() - start

print(f"\n耗时: {elapsed:.1f}s")
print(f"\n回答: {result[:500]}..." if len(result) > 500 else f"\n回答: {result}")

#### DeepResearchTool — 查看思考过程（thinking）

In [ ]:
# 使用 get_thinking_tool 获取可查看思考过程的工具
thinking_tool = dr_tool.get_thinking_tool()

query = "设备故障的主要原因有哪些？"
print(f"查询: {query}")
print("正在执行（显示思考过程）...")

result = thinking_tool._run(query)

print(f"\n=== 思考过程（前500字）===")
print(result.get("thinking_process", "")[:500])

print(f"\n=== 最终答案（前500字）===")
print(result.get("answer", "")[:500])

print(f"\n检索信息数: {len(result.get('retrieved_info', []))}")
print(f"执行日志数: {len(result.get('execution_logs', []))}")

### 4.2 DeeperResearchTool（增强版深度研究）

> 整合社区感知、动态知识图谱、Chain of Exploration 和证据链追踪。

In [ ]:
import time
from backend.search.tool.deeper_research_tool import DeeperResearchTool

dr2_tool = DeeperResearchTool()

# 可选：限制迭代轮数以加速测试
dr2_tool.deep_research.max_iterations = 2

query = "电机过热故障原因分析"
print(f"查询: {query}")
print(f"最大迭代: {dr2_tool.deep_research.max_iterations}")

start = time.time()
result = dr2_tool.search(query)
elapsed = time.time() - start

print(f"\n耗时: {elapsed:.1f}s")
print(f"\n回答: {result[:500]}..." if len(result) > 500 else f"\n回答: {result}")

#### DeeperResearchTool — 查看思考过程

In [ ]:
thinking_result = dr2_tool.thinking("电机过热故障原因分析")

print(f"=== 思考过程（前500字）===")
print(thinking_result.get("thinking_process", "")[:500])

print(f"\n=== 推理链步骤数 ===")
chain = thinking_result.get("reasoning_chain", [])
print(f"  推理步骤: {len(chain)}")

print(f"\n=== 证据来源统计 ===")
stats = thinking_result.get("evidence_stats", {})
print(f"  {stats}")

print(f"\n=== 知识图谱统计 ===")
kg = thinking_result.get("knowledge_graph", {})
print(f"  实体数: {kg.get('entity_count', 0)}")
print(f"  关系数: {kg.get('relation_count', 0)}")

print(f"\n=== 矛盾检测 ===")
contradictions = thinking_result.get("contradictions", [])
print(f"  发现 {len(contradictions)} 个矛盾")
for c in contradictions:
    print(f"    - [{c['type']}] {c.get('context', c.get('analysis', ''))[:100]}")

---
## 5. 专用工具测试

### 5.1 ChainOfExplorationTool — 链式探索

In [ ]:
from backend.search.tool.chain_exploration_tool import ChainOfExplorationTool

coe_tool = ChainOfExplorationTool(max_steps=3, exploration_width=2)

result = coe_tool.explore(
    "设备故障原因",
    start_entities=["设备", "故障"],
    max_steps=2,
)

print(f"查询: {result['query']}")
print(f"起始实体: {result['start_entities']}")

summary = result.get("summary", {})
print(f"\n探索路径:")
for p in summary.get("exploration_path", []):
    print(f"  step {p['step']}: {p.get('node_id', '')} - {p.get('reasoning', '')[:80]}")

print(f"\n统计: {summary.get('statistics', {})}")
print(f"检索结果数量: {len(result.get('retrieval_results', []))}")

### 5.2 HypothesisGeneratorTool — 假设生成

In [ ]:
from backend.search.tool.hypothesis_tool import HypothesisGeneratorTool

hyp_tool = HypothesisGeneratorTool()
hypotheses = hyp_tool.generate("导致电机过热的原因有哪些？")

print(f"生成的假设:")
for i, h in enumerate(hypotheses, 1):
    print(f"  {i}. {h}")

# 测试 Tool 封装
tool = hyp_tool.get_tool()
result = tool._run("设备振动异常的可能原因")
print(f"\nTool 返回: {result}")

### 5.3 AnswerValidationTool — 答案验证

In [ ]:
from backend.search.tool.validation_tool import AnswerValidationTool

val_tool = AnswerValidationTool()

# 测试良好答案
result_good = val_tool.validate(
    "电机过热的原因有哪些？",
    "电机过热的原因包括：1. 过载运行 2. 散热不良 3. 轴承润滑不足 4. 电压异常",
)
print("=== 良好答案验证 ===")
print(f"  passed: {result_good['validation']['passed']}")
print(f"  details: {result_good['validation']}")

# 测试劣质答案
result_bad = val_tool.validate(
    "电机过热的原因有哪些？",
    "很抱歉，我不确定这个问题。",
)
print("\n=== 劣质答案验证 ===")
print(f"  passed: {result_bad['validation']['passed']}")
print(f"  details: {result_bad['validation']}")

# 测试 Tool 封装
tool = val_tool.get_tool()
tool_result = tool._run(
    query="电机过热原因",
    answer="电机过热可能由多种因素导致，包括负载过大、散热系统故障等。",
)
print(f"\n=== Tool 封装测试 ===")
print(f"  passed: {tool_result['validation']['passed']}")

print("\n✓ AnswerValidationTool 测试完成")

---
## 6. 检索结果适配器测试

In [ ]:
from backend.search.retrieval_adapter import (
    results_from_documents,
    results_from_entities,
    results_from_relationships,
    merge_retrieval_results,
    results_to_payload,
    create_retrieval_result,
    create_retrieval_metadata,
)

# 模拟 Documents
docs = [
    type("Doc", (), {"page_content": "电机过热故障分析报告", "metadata": {"id": "chunk_001", "source": "report.pdf"}}),
    type("Doc", (), {"page_content": "轴承磨损检测标准", "metadata": {"id": "chunk_002", "source": "standard.pdf"}}),
]()

r1 = results_from_documents(docs, source="test")
print(f"from_documents: {len(r1)} 条")
for r in r1:
    print(f"  source_id={r.metadata.source_id}, granularity={r.granularity}, score={r.score:.2f}")

# 模拟 Entities
entities = [
    {"id": "电机", "description": "电动机设备", "type": "Equipment"},
    {"id": "轴承", "description": "旋转支撑部件", "type": "Component"},
]
r2 = results_from_entities(entities, source="test")
print(f"\nfrom_entities: {len(r2)} 条")

# 模拟 Relationships
rels = [
    {"start": "电机", "end": "轴承", "type": "包含", "description": "电机包含轴承"},
]
r3 = results_from_relationships(rels, source="test")
print(f"from_relationships: {len(r3)} 条")

# 测试合并
merged = merge_retrieval_results(r1, r2, r3)
print(f"\n合并后: {len(merged)} 条 (去重后)")

payload = results_to_payload(r1)
print(f"payload 格式: {type(payload)}, 长度: {len(payload)}")
print(f"  payload[0] 字段: {list(payload[0].keys()) if payload else 'empty'}")

print("\n✓ 适配器测试完成")

---
## 7. 向量工具测试

In [ ]:
import numpy as np
from backend.search.utils import VectorUtils

# 测试余弦相似度
v1 = [1.0, 2.0, 3.0]
v2 = [2.0, 4.0, 6.0]  # 与 v1 方向相同
v3 = [-1.0, -2.0, -3.0]  # 与 v1 方向相反

sim_same = VectorUtils.cosine_similarity(v1, v2)
sim_opposite = VectorUtils.cosine_similarity(v1, v3)
sim_zero = VectorUtils.cosine_similarity([0, 0, 0], v1)

print(f"相同方向: {sim_same:.4f} (期望 ~1.0)")
print(f"相反方向: {sim_opposite:.4f} (期望 ~-1.0)")
print(f"零向量:   {sim_zero:.4f} (期望 0.0)")

# 测试批量余弦相似度
query_emb = np.array([1.0, 0.0, 0.0])
embeddings = [
    np.array([1.0, 0.0, 0.0]),  # 完全相同
    np.array([0.0, 1.0, 0.0]),  # 正交
    np.array([-1.0, 0.0, 0.0]), # 相反
]

batch_sim = VectorUtils.batch_cosine_similarity(query_emb, embeddings)
print(f"\n批量相似度: {batch_sim}")

# 测试 rank_by_similarity
entities = [
    {"id": "e1", "embedding": [1.0, 0.0, 0.0]},
    {"id": "e2", "embedding": [0.0, 1.0, 0.0]},
    {"id": "e3", "embedding": [0.5, 0.5, 0.0]},
]

ranked = VectorUtils.rank_by_similarity(query_emb, entities, "embedding", top_k=2)
print(f"\n排序结果:")
for item in ranked:
    print(f"  {item['id']}: score={item.get('score', 'N/A'):.4f}")

# 测试 filter_documents_by_relevance
docs = [
    type("Doc", (), {"page_content": "电机过热故障分析", "metadata": {"id": "1"}}),
    type("Doc", (), {"page_content": "天气预报今日晴", "metadata": {"id": "2"}}),
    type("Doc", (), {"page_content": "电机维护保养指南", "metadata": {"id": "3"}}),
]()

query_vec = embeddings.embed_query("电机故障")
filtered = VectorUtils.filter_documents_by_relevance(
    query_vec, docs, embedding_func=lambda d: embeddings.embed_query(d.page_content), top_k=2
)
print(f"\n过滤结果:")
for doc in filtered:
    score = getattr(doc, "score", "N/A")
    print(f"  [{score:.4f}] {doc.page_content[:30]}..." if isinstance(score, float) else f"  {doc}")

print("\n✓ VectorUtils 测试完成")

---
## 8. 工具注册表 — 组合使用测试

In [ ]:
from backend.search.tool_registry import get_tool_class, create_extra_tool

# 组合使用：搜索 → 验证
query = "电机过热故障原因分析"

# 1. 搜索
HybridTool = get_tool_class("hybrid_search")
hybrid = HybridTool()
answer = hybrid.search(query)
print(f"搜索回答: {answer[:200]}..." if len(answer) > 200 else f"搜索回答: {answer}")

# 2. 生成假设
hypothesis_tool = create_extra_tool("hypothesis_generator")
hypotheses = hypothesis_tool.generate(query)
print(f"\n假设: {hypotheses}")

# 3. 验证答案
validator_tool = create_extra_tool("answer_validator")
validation = validator_tool.validate(query, answer)
print(f"\n验证 passed: {validation['validation']['passed']}")

# 4. 列出所有可用工具
print(f"\n可用搜索工具: {list(available_tools().keys())}")
print(f"可用专用工具: {list(available_extra_tools().keys())}")

print("\n✓ 组合使用测试完成")

---
## 9. 性能对比

In [ ]:
import time
from collections import OrderedDict

query = "设备故障原因分析"

results = OrderedDict()

# HybridSearchTool
start = time.time()
hybrid = HybridSearchTool()
ans = hybrid.search(query)
elapsed = time.time() - start
results["HybridSearchTool"] = {"time": elapsed, "answer_len": len(ans)}

# NaiveSearchTool
start = time.time()
naive = NaiveSearchTool()
ans = naive.search(query)
elapsed = time.time() - start
results["NaiveSearchTool"] = {"time": elapsed, "answer_len": len(ans)}

# LocalSearchTool
start = time.time()
ls = LocalSearchTool()
ans = ls.search(query)
elapsed = time.time() - start
results["LocalSearchTool"] = {"time": elapsed, "answer_len": len(ans)}

# GlobalSearchTool
start = time.time()
gs = GlobalSearchTool()
ans = gs.search(query)
elapsed = time.time() - start
results["GlobalSearchTool"] = {"time": elapsed, "answer_len": len(ans)}

print(f"{'工具名称':<20} {'耗时(s)':<12} {'回答长度':<10}")
print("-" * 42)
for name, r in results.items():
    print(f"{name:<20} {r['time']:<12.2f} {r['answer_len']:<10}")

---
## 10. 清理资源

In [ ]:
# 关闭所有工具连接
local.close() if 'local' in dir() else None
global_s.close() if 'global_s' in dir() else None
ls_tool.close() if 'ls_tool' in dir() else None
gs_tool.close() if 'gs_tool' in dir() else None
hybrid.close() if 'hybrid' in dir() else None
naive.close() if 'naive' in dir() else None
dr_tool.close() if 'dr_tool' in dir() else None
dr2_tool.close() if 'dr2_tool' in dir() else None

print("✓ 所有资源已关闭")